# Trade-off Sweep + Privacy Audit (Kaggle)

Two paper-critical experiments on NIH ChestX-ray14:

**Part A - Privacy-robustness-utility trade-off (regime map).** Federated AUC as the
fraction of malicious clients grows, for FedAvg and a robust rule (median), each with and
without differential privacy. Produces the regime map and a practitioner recommendation.

**Part B - Privacy audit.** A membership-inference attack and a controlled
gradient-inversion reconstruction at several DP budgets, verifying that the declared
privacy budget actually reduces leakage (not just asserted).

Scale note: both use the same *defence-sweep* scale as the earlier DP/robustness runs
(6 nodes, 128px, ~12k-image subset), which is smaller and faster than the core comparison
(3 nodes, 224px, ~30k). This is stated clearly so the two scales are never confused.

Sidebar: **GPU On, Internet On**, **+ Add Input** `nih-chest-xrays/data` (+ optionally the
224px mirror). Run All; long run (~4-6 h) - use **Save & Run All (Commit)** for unattended.


## 1. GPU + install + clone

In [ ]:
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
!pip install -q "flwr>=1.8,<2.0" "opacus>=1.4" torchmetrics scikit-learn pyyaml typer
import os
REPO_URL = "https://github.com/Pooja-Goel07/Federated-Learning-for-Privacy-Preserving-Medical-Diagnosis.git"
if not os.path.exists("/kaggle/working/fl"):
    !git clone -q $REPO_URL /kaggle/working/fl
%cd /kaggle/working/fl
!git log --oneline -1

## 2. Locate the mounted dataset

In [ ]:
import glob, os, shutil
import pandas as pd
NIH = "/kaggle/working/nih"; os.makedirs(NIH, exist_ok=True)
print("Contents of /kaggle/input:")
for d in sorted(glob.glob("/kaggle/input/*")):
    print("  ", d, sum(1 for _ in glob.iglob(d + "/**/*.png", recursive=True)), "png")
all_csv = glob.glob("/kaggle/input/**/*.csv", recursive=True)
labels = next((c for c in all_csv if os.path.basename(c).lower() == "data_entry_2017.csv"), None)
if labels is None:
    for c in all_csv:
        try:
            if {"Image Index","Finding Labels"}.issubset(pd.read_csv(c, nrows=1).columns): labels=c; break
        except Exception: pass
assert labels, "Add the 'nih-chest-xrays/data' dataset."
shutil.copy(labels, f"{NIH}/Data_Entry_2017.csv")
bb = next((c for c in all_csv if os.path.basename(c).lower()=="bbox_list_2017.csv"), None)
if bb: shutil.copy(bb, f"{NIH}/BBox_List_2017.csv")
best, best_n = None, 0
for d in sorted(glob.glob("/kaggle/input/*")):
    n = sum(1 for _ in glob.iglob(d + "/**/*.png", recursive=True))
    if n > best_n: best, best_n = d, n
IMAGE_ROOT = best
print("IMAGE_ROOT:", IMAGE_ROOT, "| png:", best_n); assert best_n > 0
import sys; sys.path.insert(0, "/kaggle/working/fl")

## 3. Config (defence-sweep scale)

In [ ]:
from src.experiments.config import load_config
cfg = load_config("configs/kaggle_defences.yaml")
cfg["data"]["root"] = NIH
cfg["data"]["image_root"] = IMAGE_ROOT
cfg["partition"]["manifest_path"] = f"{NIH}/manifest_def.json"
cfg["data"]["subsample"]["max_images"] = 12000
cfg["training"]["rounds"] = 8
print("nodes", cfg["partition"]["num_nodes"], "| imgs", cfg["data"]["subsample"]["max_images"], "| 128px | rounds", cfg["training"]["rounds"])

## 4. Part A - Trade-off sweep (regime map)

12 federated runs: attacker fraction in {0, 1/6, 2/6} x {FedAvg, median} x {no-DP, DP σ=1.0}.
Resumable: re-run to continue after a disconnect (only unfinished runs re-execute).

In [ ]:
from src.experiments.sweep import run_sweep, trade_off_variants
from src.evaluation.defence_report import generate_tradeoff_report
OUT_TO = "/kaggle/working/results/tradeoff"
variants = trade_off_variants(attack_type="model_replacement", fractions=(0.0, 0.17, 0.34),
                              dp_noise=1.0, strategies=("fedavg", "median"))
run_sweep(cfg, variants, OUT_TO, data_root=NIH)
generate_tradeoff_report(OUT_TO, robust_rule="median")
from IPython.display import Image, display, Markdown
display(Image(f"{OUT_TO}/plots/08_tradeoff_regime_map.png"))
display(Markdown(open(f"{OUT_TO}/TRADEOFF_REPORT.md").read()))

## 5. Part B - Privacy audit

Trains a no-DP model and DP models (σ=1.0, 2.0), runs the membership-inference attack on
each, and runs the controlled gradient-inversion reconstruction. Uses an **over-fitting
regime** (small ~2k subset, 25 epochs) on purpose: a membership attack only has signal
when the undefended model memorizes its training set, so we deliberately make it do so,
then show DP removes that leakage. DP runs are slow (Opacus per-sample gradients).

In [ ]:
import copy
from src.experiments.audit import run_privacy_audit
from src.evaluation.audit_report import generate_audit_report
OUT_AU = "/kaggle/working/results/audit"
audit_cfg = copy.deepcopy(cfg)
audit_cfg["data"]["subsample"]["max_images"] = 2000      # small set -> memorization -> membership leaks
audit_cfg["partition"]["manifest_path"] = f"{NIH}/manifest_audit.json"
audit_cfg["training"]["local_epochs"] = 25               # over-fit so the no-DP model leaks
run_privacy_audit(audit_cfg, OUT_AU, data_root=NIH, dp_noise_levels=(1.0, 2.0),
                  inversion_steps=300, inversion_size=32)
generate_audit_report(OUT_AU)
from IPython.display import Image, display, Markdown
display(Image(f"{OUT_AU}/plots/09_membership_inference.png"))
display(Image(f"{OUT_AU}/plots/10_gradient_inversion.png"))
display(Markdown(open(f"{OUT_AU}/AUDIT_REPORT.md").read()))

## 6. Download everything

In [ ]:
import shutil
from IPython.display import FileLink, display
shutil.make_archive("/kaggle/working/tradeoff_audit", "zip", "/kaggle/working/results")
print("zipped -> /kaggle/working/tradeoff_audit.zip")
display(FileLink("tradeoff_audit.zip"))